# KilnWatch — baseline kiln detector

Before running, in the notebook sidebar:
1. **Add Input** → search `sentinelkiln-dataset` (by rishabhsnip) → add it.
2. **Settings → Accelerator** → GPU T4 x2 (or P100).
3. **Settings → Internet** → on (to install Ultralytics and clone the repo).

Run top to bottom. The baseline is a pipeline check (are labels, paths and scores sane?), not the final accuracy claim.

In [ ]:
!pip install -q ultralytics
!git clone -q https://github.com/aryamanraj2/KilnWatch.git

## 1. Prepare the data
Images are symlinked from the read-only input, so this takes no extra disk. Labels go to `/tmp/kilns` to keep them out of the notebook's saved output.

In [ ]:
from pathlib import Path
SRC = next(p for pat in ("*/train/images", "*/*/train/images", "*/*/*/train/images")
           for p in Path("/kaggle/input").glob(pat)).parents[1]
print(SRC)
!python KilnWatch/scripts/prepare_data.py --src "{SRC}" --dst /tmp/kilns --link

Check the labels by eye before training: boxes should sit on the reddish kiln footprints, and the type names should match the shape (FCBK long oval, CFCBK round, Zigzag rectangle).

In [ ]:
!python KilnWatch/scripts/show_labels.py /tmp/kilns/train --n 20 --out labels_check.png
from IPython.display import Image
Image("labels_check.png")

## 2. Baseline: yolo11s-obb, 128 px, 20 epochs, official split
Scores come out for two test sets:
- **full**: every test tile. Compare `any_kiln_AP50` with the paper's YOLO11-L OBB (75.2). A small model at 20 epochs should land below that but in the same region; a near-zero score means a pipeline bug.
- **strict**: the NCR tiles. Most of these tiles share ground with training tiles in this split, so expect this score to be optimistic.

In [ ]:
!python KilnWatch/scripts/train.py --data /tmp/kilns/kilns_full.yaml --model yolo11s-obb.pt --epochs 20 --name baseline --cache ram

In [ ]:
import json, glob
# Ultralytics may nest runs (e.g. runs/obb/runs/kilnwatch/...), so find the run instead of guessing.
def run_dir(name):
    return Path(sorted(glob.glob(f"runs/**/kilnwatch/{name}/scores.json", recursive=True))[-1]).parent
base = run_dir("baseline")
print(base)
print(json.dumps(json.load(open(base / "scores.json"))["test"], indent=2))
Image(str(base / "results.png"))

## 3. Strict NCR score (run once the baseline looks right)
The same model trained without any tile that shares ground with an NCR test tile. Its **strict** score is the honest answer to "how well does this work in our area?". The gap from the baseline's strict score shows how much the official split flatters it.

In [ ]:
!python KilnWatch/scripts/train.py --data /tmp/kilns/kilns_strict.yaml --model yolo11s-obb.pt --epochs 20 --name baseline_strict --cache ram
print(json.dumps(json.load(open(run_dir("baseline_strict") / "scores.json"))["test"], indent=2))

## 4. Keep the results
The run folders (under `runs/`, found by `run_dir`) hold the weights, curves and `scores.json`. **Save Version** keeps them as notebook output. To push them to S3, add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` under **Add-ons → Secrets** and run the cell below.

In [ ]:
# import os
# from kaggle_secrets import UserSecretsClient
# s = UserSecretsClient()
# os.environ["AWS_ACCESS_KEY_ID"] = s.get_secret("AWS_ACCESS_KEY_ID")
# os.environ["AWS_SECRET_ACCESS_KEY"] = s.get_secret("AWS_SECRET_ACCESS_KEY")
# os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
# from KilnWatch.scripts.train import upload   # or: aws s3 cp --recursive
# for name in ("baseline", "baseline_strict"):
#     upload(run_dir(name), "s3://YOUR-BUCKET/models", name)